# 🏆 Hệ thống Nhận diện Rác 2 Giai đoạn (Final Pipeline)

Notebook này thực thi **TOÀN BỘ** quy trình của đồ án từ đầu đến cuối trên Kaggle.

### 📐 Kiến trúc đã chốt:
1. **Stage 1 (Phát hiện vùng rác):** Dùng **YOLOv8s** gốc, đặt `conf=0.15` để quét và lấy bằng được mọi thứ nghi ngờ là rác (Tối đa hóa Recall).
2. **Stage 2 (Phân loại & Lọc):** Dùng **EfficientNet-B2** với 6 lớp (Thêm lớp `Background` để lọc các cú lừa từ Stage 1).

⚠️ **Yêu cầu hệ thống:** GPU (T4 x2 hoặc P100) + Bật Internet.

In [ ]:
# ============================================================
# 1. Tải Mã nguồn & Cài đặt thư viện
# ============================================================
!git clone https://github.com/Shiba-dotcom/waste-detection2-Stage.git
!pip install -q ultralytics timm

In [ ]:
# ============================================================
# 2. Nhập Dữ liệu Ngoại lai (TACO, TrashNet, RealWaste)
# ============================================================
import os, shutil

!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/TrashNet
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/RealWaste
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

datasets_to_copy = [
    {"src": "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/TrashNet"},
    {"src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset/data",
     "dst": "/kaggle/working/waste-detection2-Stage/data/raw"},
    {"src": "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/RealWaste"}
]

for task in datasets_to_copy:
    if os.path.exists(task["src"]):
        os.makedirs(task["dst"], exist_ok=True)
        shutil.copytree(task["src"], task["dst"], dirs_exist_ok=True)
        print(f"Đã tải: {os.path.basename(task['src'])}")
    else:
        print(f"Bỏ qua: {task['src']} (Không tìm thấy trên Kaggle Dataset)")

In [ ]:
# ============================================================
# 3. Chạy Toàn bộ Tiền xử lý Dữ liệu (Data Pipeline)
# ============================================================
%cd /kaggle/working/waste-detection2-Stage

print("\n--- 3.1 Dọn dẹp & Tạo nhãn YOLO ---")
!python src/data_prep/data_cleaning.py
!python src/Training_dataYolo.py
!python src/data_prep/split_dataset.py

print("\n--- 3.2 Chuẩn bị dữ liệu cho Stage 2 (Classifier) ---")
!python src/data_prep/crop_for_classification.py
!python src/data_prep/merge_external_datasets.py
!python src/data_prep/generate_background.py

print("\n✅ Hoàn tất chuẩn bị 100% dữ liệu!")

In [ ]:
# ============================================================
# 4. Huấn luyện Stage 1 (YOLOv8s - Binary Detector)
# BỎ QUA CELL NÀY NẾU BẠN ĐÃ CÓ FILE stage1_best.pt
# ============================================================
from ultralytics import YOLO
import time
from pathlib import Path

DATASET_YAML = "/kaggle/working/waste-detection2-Stage/data/processed_binary/dataset.yaml"
OUTPUT_DIR   = "/kaggle/working/waste-detection2-Stage/results/yolo8s_runs"

# Kiểm tra xem đã có model train sẵn chưa để bỏ qua
if not Path("stage1_best.pt").exists():
    print("🚀 Bắt đầu train Stage 1 (YOLOv8s)")
    model_s = YOLO("yolov8s.pt")
    t0 = time.time()
    model_s.train(
        data=DATASET_YAML, imgsz=640, epochs=100, batch=16, patience=20,
        optimizer="auto", lr0=0.01, cos_lr=True, augment=True, workers=4,
        project=OUTPUT_DIR, name="stage1", exist_ok=True, save=True, save_period=-1,
    )
    print(f"\n✅ Train Stage 1 hoàn tất sau {(time.time()-t0)/60:.1f} phút")
    
    # Copy model tốt nhất ra thư mục gốc để dễ dùng
    !cp results/yolo8s_runs/stage1/weights/best.pt stage1_best.pt
else:
    print("⏭️ Đã tìm thấy 'stage1_best.pt', bỏ qua bước huấn luyện Stage 1.")

In [ ]:
# ============================================================
# 5. Huấn luyện Stage 2 (EfficientNet-B2 - 6 Lớp Classifier)
# ============================================================
# Train với các cơ chế: Freeze/Unfreeze, Dropout 0.4, RandAugment, Class Weights

!python src/train_stage2_classifier.py

In [ ]:
# ============================================================
# 6. Đánh giá Toàn Trình (End-to-End 2-Stage Pipeline)
# ============================================================
# Bước này mô phỏng luồng chạy thực tế: Đưa ảnh vào YOLO -> Cắt rác -> Phân loại

!python src/evaluate_2stage.py \
  --detector stage1_best.pt \
  --classifier models/stage2_best.pth \
  --data-dir data/processed_binary/images/test \
  --label-dir data/processed/labels/test \
  --conf 0.10 \
  --output results/2stage_eval

print("\n✅ ĐÁNH GIÁ HOÀN TẤT! Kết quả được lưu tại results/2stage_eval")

In [ ]:
# ============================================================
# 7. Zip tất cả kết quả để tải về
# ============================================================
import shutil

# Nén thư mục kết quả
shutil.make_archive("/kaggle/working/final_results", "zip", "/kaggle/working/waste-detection2-Stage/results")
print("\n📦 Đã nén toàn bộ logs, biểu đồ và kết quả: /kaggle/working/final_results.zip")

# Nén weights của model
if os.path.exists("/kaggle/working/waste-detection2-Stage/models"):
    shutil.make_archive("/kaggle/working/final_models", "zip", "/kaggle/working/waste-detection2-Stage/models")
    print("📦 Đã nén trọng số mô hình: /kaggle/working/final_models.zip")